# SayIt: Dataset v2 dan Training Ulang Deteksi Uang Rupiah

Notebook ini menyusun **dataset v2** dari dua dataset publik, lalu melatih ulang model.

**Kenapa perlu v2** (hasil analisis notebook 01):
- Dataset A (`deteksi-rupiah-jkkmu-zagcv`) berisi campuran label yang benar dan label yang tertukar, 3 salinan augmentasi per foto, dan kebocoran antar split. Label yang tertukar tidak bisa diperbaiki dengan satu aturan global.
- Dataset B bersih: 1.125 foto unik, dan nominal di nama file 100% cocok dengan label.

**Aturan penyusunan v2:**
1. Dataset B dipakai seluruhnya. Gambar satu-objek yang nominal di nama filenya tidak cocok dengan label dibuang.
2. Dari dataset A, hanya diambil gambar **satu-objek yang nominalnya tertulis di nama file**, dengan label diganti sesuai nama file. Gambar A tanpa objek sama sekali juga diambil sebagai contoh latar. Sisanya dibuang karena labelnya tidak bisa diverifikasi.
3. Setiap foto asal hanya diambil satu salinan.
4. Foto yang sangat mirip dikelompokkan, lalu satu kelompok selalu masuk ke split yang sama (tidak bocor).
5. Pembagian 70/20/10, distratifikasi per nominal.

**Perubahan v2.1** (dari hasil v2): kotak bagian uang yang bersarang di dalam kotak uang utama dibuang, dan ditambahkan 120 contoh latar tanpa uang dari COCO128.

**Sebelum mulai:** runtime **T4 GPU**, dan secret `ROBOFLOW_API_KEY` sudah aktif untuk notebook ini.

## 1. Persiapan

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics roboflow imagehash

In [ ]:
import json, re, random, shutil
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image as IPImage, display
from PIL import Image
from roboflow import Roboflow
from ultralytics import YOLO

DRIVE_DIR = Path('/content/drive/MyDrive/sayit')
RUNS_DIR = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
DATA_DIR = DRIVE_DIR / 'datasets'
for d in (RUNS_DIR, MODELS_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Konfigurasi

In [ ]:
# ====== ISI BAGIAN INI ======
# Dataset dari Roboflow. Tambah sebanyak yang perlu, urutan tidak penting.
#   mode 'label_dari_nama' : label hanya dipakai bila nominal tertulis di nama berkas, dan label
#                            diganti mengikuti nama itu. Untuk dataset A yang sebagian labelnya tertukar.
#   mode 'percaya_label'   : label dipakai apa adanya, tetapi gambar dibuang bila labelnya
#                            bertentangan dengan nominal di nama berkas.
#   sendiri=True           : foto tim sendiri. SELURUHNYA masuk split test, tidak pernah ikut latih.
DATASETS = [
    dict(nama='A', workspace='mochammadalthofsr-gmail-com',
         project='deteksi-rupiah-jkkmu-zagcv', versi=1, mode='label_dari_nama'),
    dict(nama='B', workspace='mochammadalthofsr-gmail-com',
         project='isi-slug-project-dataset-B', versi=1, mode='percaya_label'),

    # Tambahan dataset publik. Pilih versi TANPA augmentasi, dan pastikan nama kelas di Roboflow
    # sudah persis '1000' ... '100000' (sel unduh berhenti kalau ada nama lain).
    # dict(nama='D', workspace='...', project='...', versi=1, mode='percaya_label'),
    # dict(nama='E', workspace='...', project='...', versi=1, mode='percaya_label'),

    # Foto sendiri yang sudah dilabeli. Selama baris ini masih dikomentari, split test memakai
    # data publik dan angkanya TIDAK mewakili dunia nyata.
    # Dua cara, pilih salah satu:
    #   a) folder di Google Drive (hasil scripts/buat_label_uang.py, tinggal disalin ke Drive):
    # dict(nama='SENDIRI', lokal='/content/drive/MyDrive/sayit/data/label_sendiri',
    #      mode='percaya_label', sendiri=True),
    #   b) diunggah ke Roboflow seperti dataset lain:
    # dict(nama='SENDIRI', workspace='...', project='...', versi=1, mode='percaya_label', sendiri=True),
]

CLASSES = ['1000', '2000', '5000', '10000', '20000', '50000', '100000']

SPLIT_RATIO = (0.7, 0.2, 0.1)   # train, val, test. Bagian test diabaikan bila ada dataset sendiri=True
SEED = 42
PHASH_MAX_DIST = 8              # jarak Hamming maksimum agar dua foto dianggap "mirip"

MODEL_WEIGHTS = 'yolov8n.pt'
EPOCHS   = 100
PATIENCE = 20
IMGSZ    = 640
BATCH    = 16
RUN_NAME = 'money_v2_3'         # ganti tiap percobaan agar run sebelumnya tidak tertimpa
CONF_THRESHOLD = 0.5

# Augmentasi. Yang tidak disebut di sini memakai bawaan ultralytics.
# Alasan tiap angka ada di sel penjelasan sebelum training.
AUG = dict(
    fliplr=0.0,          # uang cermin tidak pernah ada di dunia nyata
    degrees=180,         # WAJIB: pengguna tunanetra tidak bisa membidik kamera
    perspective=0.0005,  # difoto dari sudut, bukan tegak lurus di atas meja
    scale=0.6,           # jarak kamera ke uang sangat bervariasi
    translate=0.2,       # uang tidak selalu di tengah bingkai
    hsv_v=0.5,           # lampu kuning, siang terang, bayangan tangan
    hsv_s=0.8,
)

# Anotasi bagian uang (potret, angka, benang pengaman) yang berada di dalam kotak uang utama dibuang
NESTED_CONTAIN = 0.85           # kotak dibuang jika >= 85% luasnya berada di dalam kotak lain berkelas sama

# Contoh latar tanpa uang dari COCO128 untuk mengurangi deteksi palsu
ADD_COCO_BACKGROUNDS = True
N_BACKGROUNDS = 120
COCO128_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip'

# Model pembanding pada split test yang sama.
BASELINE_PT = RUNS_DIR / 'money_v2_2' / 'weights' / 'best.pt'
# ============================

random.seed(SEED)
np.random.seed(SEED)

def ambil_kunci_roboflow() -> str:
    """Ambil API key Roboflow, tahan terhadap perbedaan versi Colab.

    Sebagian versi `google.colab.userdata.get()` mengembalikan str, sebagian
    mengembalikan dict. Kalau keduanya gagal, kunci diminta lewat ketikan.
    """
    kunci = None
    try:
        from google.colab import userdata
        kunci = userdata.get('ROBOFLOW_API_KEY')
    except Exception as e:
        print('Secret ROBOFLOW_API_KEY tidak terbaca:', type(e).__name__)
    if isinstance(kunci, dict):
        kunci = (kunci.get('ROBOFLOW_API_KEY') or kunci.get('value')
                 or next((v for v in kunci.values() if isinstance(v, str)), None))
    if not isinstance(kunci, str) or not kunci.strip():
        from getpass import getpass
        kunci = getpass('Tempel Roboflow API key (Settings > API Keys di roboflow.com): ')
    return kunci.strip()


RF_KEY = ambil_kunci_roboflow()
assert RF_KEY, 'API key kosong.'
rf = Roboflow(api_key=RF_KEY)


## 3. Siapkan semua dataset

Tiap entri `DATASETS` boleh berasal dari **Roboflow** (`workspace` + `project` + `versi`) atau dari
**folder lokal di Google Drive** (`lokal='/content/drive/MyDrive/...'`). Drive sudah ter-mount di
Bagian 1, jadi folder apa pun di sana bisa dibaca langsung tanpa lewat Roboflow.

Folder lokal harus berbentuk dataset YOLO: `<root>/test/images/*.jpg` dan `<root>/test/labels/*.txt`,
dengan `data.yaml` berisi daftar nama kelas. Bentuk itu persis yang dihasilkan
`scripts/buat_label_uang.py` di laptop, jadi tinggal salin foldernya ke Drive.

Sel ini berhenti keras bila ada nama kelas yang tidak dikenal. Itu disengaja: nama kelas yang
berbeda antar-dataset (`Rp1000`, `1k`, `1000 rupiah`) adalah sumber salah label yang paling mudah
lolos tanpa disadari. Rapikan namanya menjadi `1000` sampai `100000`, lalu ulangi.


In [ ]:
def dari_roboflow(d):
    ds = rf.workspace(d['workspace']).project(d['project']).version(d['versi']).download(
        'yolov8', location=f"/content/src_{d['nama']}", overwrite=True)
    return Path(ds.location)


def baca_names(root: Path):
    """Daftar nama kelas dari data.yaml. Tanpa data.yaml, urutan CLASSES yang dipakai."""
    yml = root / 'data.yaml'
    if not yml.exists():
        print(f'  (tanpa data.yaml, dianggap memakai urutan CLASSES)')
        return list(CLASSES)
    with open(yml) as f:
        names = yaml.safe_load(f)['names']
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]
    return [str(n) for n in names]


def siapkan(d):
    """Root dan nama kelas satu dataset, dari Roboflow atau dari folder Drive."""
    if d.get('lokal'):
        root = Path(d['lokal'])
        assert root.exists(), f"{d['nama']}: folder tidak ada: {root}"
        asal = f'lokal {root}'
    else:
        root = dari_roboflow(d)
        asal = f"roboflow {d['project']}"
    names = baca_names(root)
    unknown = set(names) - set(CLASSES)
    assert not unknown, f"{d['nama']}: kelas tidak dikenal {unknown}. Rapikan nama kelas dulu."
    # dihitung di sini, bukan lewat image_files() dari Bagian 4, supaya sel ini tidak
    # bergantung pada urutan menjalankan dan Runtime > Run all tetap aman
    n_gambar = sum(1 for split in ('train', 'valid', 'val', 'test')
                   for p in (root / split / 'images').glob('*') if p.is_file())
    assert n_gambar, f"{d['nama']}: tidak ada gambar di {root}. Cek strukturnya: <root>/test/images/"
    print(f"{d['nama']:10s} {asal:55s} {n_gambar:>5d} gambar, kelas {names}")
    return root, names


for d in DATASETS:
    d['root'], d['names'] = siapkan(d)

punya_sendiri = [d['nama'] for d in DATASETS if d.get('sendiri')]
print()
if punya_sendiri:
    print('Dataset foto sendiri:', ', '.join(punya_sendiri), '-> seluruhnya menjadi split test.')
else:
    print('PERINGATAN: belum ada dataset foto sendiri (sendiri=True).')
    print('            Split test akan berasal dari sumber yang sama dengan data latih, sehingga')
    print('            mAP di Bagian 10 optimistis dan tidak mewakili dunia nyata.')
    print('            Ukur yang sebenarnya di laptop:')
    print(r'              python scripts\diagnosa_uang.py data\uji_sendiri --harapan-dari-nama')


## 4. Fungsi bantu

Membaca label (poligon diubah menjadi kotak, anotasi ganda dibuang), membaca nominal dari nama file, dan menentukan foto asal.


In [ ]:
K_RE = re.compile(r'(?i)(?<!\d)(1|2|5|10|20|50|100)\s*k{1,2}(?![a-z])')
N_RE = re.compile(r'(?<!\d)(1000|2000|5000|10000|20000|50000|100000)(?!\d)')

def nominal_from_name(name):
    m = K_RE.search(name)
    if m:
        return str(int(m.group(1)) * 1000)
    m = N_RE.search(name)
    return m.group(1) if m else None

def source_name(img: Path):
    return img.name.split('.rf.')[0]

def xywh_to_xyxy(a):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2,
                     a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], axis=1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def read_labels(lbl_path: Path, names, dup_iou=0.7):
    """Kembalikan list (nama_kelas, [xc, yc, w, h]) tanpa duplikat."""
    items = []
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            vals = list(map(float, parts[1:]))
            if len(vals) != 4:
                xs, ys = vals[0::2], vals[1::2]
                vals = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
            items.append((names[int(parts[0])], vals))
    kept = []
    for c, box in items:
        bx = xywh_to_xyxy(box)
        if not any(c == kc and iou_matrix(bx, xywh_to_xyxy(kb))[0, 0] > dup_iou for kc, kb in kept):
            kept.append((c, box))
    return kept

def nested_mask(classes, xyxy, contain_thr=0.85):
    """True untuk kotak yang >= contain_thr luasnya berada di dalam kotak LAIN yang lebih besar dan berkelas sama."""
    xyxy = np.asarray(xyxy, dtype=float).reshape(-1, 4)
    area = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    mask = np.zeros(len(xyxy), dtype=bool)
    for i in range(len(xyxy)):
        for j in range(len(xyxy)):
            if i == j or classes[i] != classes[j] or area[j] <= area[i]:
                continue
            iw = min(xyxy[i, 2], xyxy[j, 2]) - max(xyxy[i, 0], xyxy[j, 0])
            ih = min(xyxy[i, 3], xyxy[j, 3]) - max(xyxy[i, 1], xyxy[j, 1])
            if iw > 0 and ih > 0 and iw * ih / (area[i] + 1e-9) >= contain_thr:
                mask[i] = True
                break
    return mask

def drop_nested(items, contain_thr=0.85):
    if len(items) < 2:
        return items, 0
    mask = nested_mask([c for c, _ in items], xywh_to_xyxy([b for _, b in items]), contain_thr)
    return [it for it, m in zip(items, mask) if not m], int(mask.sum())

def image_files(root: Path):
    for split in ('train', 'valid', 'val', 'test'):
        d = root / split / 'images'
        if d.exists():
            for p in sorted(d.iterdir()):
                if p.is_file():
                    yield p

def black_fraction(p: Path):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    return float((im < 10).mean()) if im is not None else 1.0

def phash_bits(p: Path):
    with Image.open(p) as im:
        return imagehash.phash(im.convert('RGB')).hash.flatten()

## 5. Seleksi data

In [ ]:
records = []
log = Counter()
nested_examples = []


def tambah_percaya_label(d):
    """Label dipakai apa adanya, kecuali bertentangan dengan nominal di nama berkas."""
    for img in image_files(d['root']):
        labels = read_labels(img.parent.parent / 'labels' / f'{img.stem}.txt', d['names'])
        labels, n_nested = drop_nested(labels, NESTED_CONTAIN)
        if n_nested:
            log[f"{d['nama']} kotak bersarang dibuang"] += n_nested
            nested_examples.append((img, d['names']))
        src = source_name(img)
        nom = nominal_from_name(src)
        if nom and len(labels) == 1 and labels[0][0] != nom:
            log[f"{d['nama']} dibuang: nama != label"] += 1
            continue
        records.append({'dataset': d['nama'], 'path': img, 'asal': f"{d['nama']}/{src}",
                        'labels': labels, 'sendiri': bool(d.get('sendiri'))})
        log[f"{d['nama']} dipakai"] += 1


def tambah_label_dari_nama(d):
    """Hanya gambar satu objek yang nominalnya tertulis di nama berkas, plus gambar tanpa objek.

    Label asli diabaikan dan diganti sesuai nama berkas, karena dataset ini terbukti punya label
    tertukar. Satu salinan per foto asal: dipilih yang bingkai hitamnya paling sedikit, sehingga
    hasil rotasi Roboflow tidak ikut terbawa.
    """
    kandidat = defaultdict(list)
    for img in image_files(d['root']):
        labels = read_labels(img.parent.parent / 'labels' / f'{img.stem}.txt', d['names'])
        src = source_name(img)
        nom = nominal_from_name(src)
        if len(labels) == 0:
            kandidat[src].append((img, []))
        elif nom and len(labels) == 1:
            kandidat[src].append((img, [(nom, labels[0][1])]))
        else:
            log[f"{d['nama']} dibuang: tidak bisa diverifikasi"] += 1
    for src, copies in kandidat.items():
        img, labels = min(copies, key=lambda t: black_fraction(t[0]))
        records.append({'dataset': d['nama'], 'path': img, 'asal': f"{d['nama']}/{src}",
                        'labels': labels, 'sendiri': bool(d.get('sendiri'))})
        log[f"{d['nama']} dipakai (latar)" if not labels
            else f"{d['nama']} dipakai (label dari nama file)"] += 1
        log[f"{d['nama']} salinan dibuang"] += len(copies) - 1


CARA = {'percaya_label': tambah_percaya_label, 'label_dari_nama': tambah_label_dari_nama}
for d in DATASETS:
    assert d['mode'] in CARA, f"mode tidak dikenal: {d['mode']}"
    CARA[d['mode']](d)

for k, v in sorted(log.items()):
    print(f'{k:48s}: {v}')
print(f'Total gambar: {len(records)}')
print('Per dataset :', dict(Counter(r['dataset'] for r in records)))


### 5.1 Cek gambar yang kotak bersarangnya dibuang

Beberapa foto di dataset B memberi kotak terpisah untuk bagian uang (potret, angka nominal, benang pengaman). Model jadi belajar menghitung satu lembar sebagai beberapa lembar. Gambar di bawah menampilkan **kotak yang tersisa** setelah penyaringan. Pastikan setiap uang tetap punya satu kotak, dan tidak ada uang asli yang ikut hilang.

In [ ]:
show = nested_examples[:12]
print(f'{len(nested_examples)} gambar terdampak, ditampilkan {len(show)}')
if show:
    fig, axes = plt.subplots((len(show) + 3) // 4, 4, figsize=(16, 4 * ((len(show) + 3) // 4)))
    axes = np.atleast_1d(axes).flatten()
    for ax, (p, names) in zip(axes, show):
        items, _ = drop_nested(read_labels(p.parent.parent / 'labels' / f'{p.stem}.txt', names), NESTED_CONTAIN)
        im = cv2.imread(str(p)); h, w = im.shape[:2]
        for c, (xc, yc, bw, bh) in items:
            x1, y1, x2, y2 = int((xc - bw / 2) * w), int((yc - bh / 2) * h), int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            cv2.rectangle(im, (x1, y1), (x2, y2), (0, 200, 0), 3)
            cv2.putText(im, c, (x1 + 4, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2)
        ax.imshow(im[:, :, ::-1]); ax.set_title(p.name[:30], fontsize=8); ax.axis('off')
    for ax in axes[len(show):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

### 5.2 Tambah contoh latar tanpa uang (COCO128)

Dataset B tidak punya foto tanpa uang, sehingga model mudah mendeteksi uang palsu pada kain merah, kardus, dan sejenisnya. Sel ini menambahkan gambar acak dari COCO128 (foto umum tanpa uang) sebagai contoh latar dengan label kosong.

In [ ]:
import urllib.request, zipfile
if ADD_COCO_BACKGROUNDS:
    try:
        zpath = Path('/content/coco128.zip')
        if not zpath.exists():
            urllib.request.urlretrieve(COCO128_URL, zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall('/content')
        coco_imgs = sorted(Path('/content/coco128/images/train2017').glob('*.jpg'))
        picked = random.Random(SEED).sample(coco_imgs, k=min(N_BACKGROUNDS, len(coco_imgs)))
        for p in picked:
            records.append({'dataset': 'LATAR', 'path': p, 'asal': 'LATAR/' + p.stem,
                            'labels': [], 'sendiri': False})
        print(f'{len(picked)} gambar latar COCO128 ditambahkan. Total gambar v2: {len(records)}')
    except Exception as e:
        print('Gagal mengunduh COCO128, lanjut tanpa latar tambahan:', e)

## 6. Kelompokkan foto yang mirip

Foto berurutan dari sesi pemotretan yang sama bisa sangat mirip. Foto seperti itu dikelompokkan agar tidak terpisah ke train dan test. Jika ada kelompok yang terlalu besar, ambang kemiripan diturunkan otomatis.

In [ ]:
H = np.stack([phash_bits(r['path']) for r in records])
N = len(records)

def group_with(max_dist):
    parent = list(range(N))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i in range(0, N, 200):
        d = (H[i:i + 200, None, :] != H[None, :, :]).sum(-1)
        for a, b in zip(*np.nonzero(d <= max_dist)):
            a += i
            if a < b:
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[ra] = rb
    return [find(i) for i in range(N)]

dist = PHASH_MAX_DIST
while True:
    groups = group_with(dist)
    sizes = Counter(groups)
    largest = max(sizes.values())
    print(f'jarak <= {dist}: {len(sizes)} kelompok, kelompok terbesar {largest} gambar')
    if largest <= max(10, 0.03 * N) or dist <= 2:
        break
    dist -= 2

for r, g in zip(records, groups):
    r['grup'] = g
print('Sebaran ukuran kelompok:', dict(sorted(Counter(sizes.values()).items())))

## 7. Bagi train/val/test

Aturannya berbeda tergantung ada atau tidaknya dataset `sendiri=True`.

**Bila ada foto sendiri:** seluruh kelompok yang memuat foto sendiri menjadi split **test**, dan data
publik dibagi ke train/val saja. Dengan begitu angka di Bagian 10 mengukur pemakaian nyata, bukan
kemiripan dengan data latih.

**Bila belum ada:** pembagian mengikuti `SPLIT_RATIO` seperti sebelumnya, dan angkanya optimistis.
Bukti bahwa ini bukan kekhawatiran teoretis: antara v2.1 dan v2.2, augmentasi rotasi hampir tidak
mengubah mAP split test (0,966 menjadi 0,967), padahal pada foto sendiri akurasinya naik dari 20%
ke 40% dan tebakannya berhenti berubah-ubah saat foto diputar. Split test yang sesumber dengan data
latih tidak bisa melihat perbaikan yang paling penting.

Pembagian tetap dilakukan **per kelompok kemiripan pHash**, bukan per gambar, sehingga foto yang
sama muncul di dua dataset tidak akan terpisah ke train dan test.


In [ ]:
def main_class(rs):
    c = Counter(lbl for r in rs for lbl, _ in r['labels'])
    return c.most_common(1)[0][0] if c else 'latar'

by_group = defaultdict(list)
for r in records:
    by_group[r['grup']].append(r)

# Kelompok yang memuat foto sendiri menjadi test seluruhnya.
grup_sendiri = {g for g, rs in by_group.items() if any(r['sendiri'] for r in rs)}
ada_sendiri = bool(grup_sendiri)

by_class = defaultdict(list)
for g, rs in by_group.items():
    if g not in grup_sendiri:
        by_class[main_class(rs)].append(g)

rng = random.Random(SEED)
split_of_group = {g: 'test' for g in grup_sendiri}
for cls, gs in sorted(by_class.items()):
    rng.shuffle(gs)
    n = len(gs)
    n_val = max(1, round(n * SPLIT_RATIO[1])) if n >= 3 else 0
    n_test = 0 if ada_sendiri else (max(1, round(n * SPLIT_RATIO[2])) if n >= 3 else 0)
    for k, g in enumerate(gs):
        split_of_group[g] = 'test' if k < n_test else ('val' if k < n_test + n_val else 'train')

for r in records:
    r['split'] = split_of_group[r['grup']]

# Pemeriksaan kebocoran
for key in ('asal', 'grup'):
    per = defaultdict(set)
    for r in records:
        per[r[key]].add(r['split'])
    leaks = sum(len(s) > 1 for s in per.values())
    assert leaks == 0, f'Kebocoran: {leaks} {key} ada di lebih dari satu split'
print('Tidak ada kebocoran berdasarkan foto asal maupun kelompok kemiripan.')

if ada_sendiri:
    asing = [r for r in records if r['split'] == 'test' and not r['sendiri']]
    assert not asing, f'{len(asing)} gambar bukan foto sendiri ikut masuk split test'
    print(f"Split test berisi HANYA foto sendiri: {sum(r['split'] == 'test' for r in records)} gambar.")
else:
    print('PERINGATAN: split test berasal dari data publik yang sesumber dengan data latih.')
    print('            mAP di Bagian 10 optimistis. Angka sebenarnya diukur di laptop dengan')
    print(r'            python scripts\diagnosa_uang.py data\uji_sendiri --harapan-dari-nama')

nama_ds = [d['nama'] for d in DATASETS] + ['LATAR']
rows = []
for split in ('train', 'val', 'test'):
    rs = [r for r in records if r['split'] == split]
    cnt = Counter(lbl for r in rs for lbl, _ in r['labels'])
    rows.append({'split': split, 'gambar': len(rs),
                 **{f'dari_{n}': sum(r['dataset'] == n for r in rs) for n in nama_ds},
                 'tanpa_objek': sum(not r['labels'] for r in rs),
                 **{c: cnt.get(c, 0) for c in CLASSES}})
split_stats = pd.DataFrame(rows).set_index('split')
display(split_stats)


## 8. Tulis dataset v2 dan simpan ke Drive

In [ ]:
V2_ROOT = Path('/content/data_v2')
if V2_ROOT.exists():
    shutil.rmtree(V2_ROOT)
for split in ('train', 'val', 'test'):
    (V2_ROOT / split / 'images').mkdir(parents=True)
    (V2_ROOT / split / 'labels').mkdir(parents=True)

manifest = []
for i, r in enumerate(records):
    stem = f"{r['dataset']}_{i:05d}"
    dst = V2_ROOT / r['split'] / 'images' / f"{stem}{r['path'].suffix}"
    shutil.copy(r['path'], dst)
    lines = [f'{CLASSES.index(c)} ' + ' '.join(f'{v:.6f}' for v in box) for c, box in r['labels']]
    (V2_ROOT / r['split'] / 'labels' / f'{stem}.txt').write_text('\n'.join(lines))
    manifest.append({'file_v2': dst.name, 'split': r['split'], 'dataset': r['dataset'], 'asal': r['asal'],
                     'grup': r['grup'], 'label': ','.join(c for c, _ in r['labels'])})

V2_YAML = V2_ROOT / 'data.yaml'
with open(V2_YAML, 'w') as f:
    yaml.safe_dump({'path': str(V2_ROOT),
                    'train': str(V2_ROOT / 'train' / 'images'),
                    'val': str(V2_ROOT / 'val' / 'images'),
                    'test': str(V2_ROOT / 'test' / 'images'),
                    'names': {i: c for i, c in enumerate(CLASSES)}}, f, sort_keys=False)

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(DATA_DIR / 'money_v2_manifest.csv', index=False)
split_stats.to_csv(DATA_DIR / 'money_v2_split_stats.csv')
shutil.make_archive(str(DATA_DIR / 'money_v2'), 'zip', V2_ROOT)
print('Dataset v2 ditulis ke', V2_ROOT)
print('Cadangan zip, manifest, dan statistik disimpan di', DATA_DIR)

### 8.1 Pemulihan jika sesi terputus

Jika runtime mati setelah Bagian 8 selesai, jalankan Bagian 1 lalu sel di bawah ini (hapus tanda `#`), kemudian langsung ke Bagian 9.

In [ ]:
# import shutil, yaml
# from pathlib import Path
# V2_ROOT = Path('/content/data_v2')
# shutil.unpack_archive(str(DATA_DIR / 'money_v2.zip'), V2_ROOT)
# V2_YAML = V2_ROOT / 'data.yaml'
# cfg = yaml.safe_load(open(V2_YAML)); cfg['path'] = str(V2_ROOT)
# for k in ('train', 'val', 'test'): cfg[k] = str(V2_ROOT / k / 'images')
# yaml.safe_dump(cfg, open(V2_YAML, 'w'), sort_keys=False)

### 8.2 Cek visual

Periksa bahwa kotak dan nama kelas cocok dengan uangnya. Jika banyak yang salah, **berhenti** dan jangan lanjut ke training.

In [ ]:
samples = random.Random(SEED).sample(
    [p for p in (V2_ROOT / 'train' / 'images').iterdir()], k=min(12, len(list((V2_ROOT / 'train' / 'images').iterdir()))))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axes.flatten(), samples):
    im = cv2.imread(str(p)); h, w = im.shape[:2]
    for line in (p.parent.parent / 'labels' / f'{p.stem}.txt').read_text().splitlines():
        c, xc, yc, bw, bh = line.split()
        x1, y1 = int((float(xc) - float(bw) / 2) * w), int((float(yc) - float(bh) / 2) * h)
        x2, y2 = int((float(xc) + float(bw) / 2) * w), int((float(yc) + float(bh) / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 200, 0), 3)
        cv2.putText(im, CLASSES[int(c)], (x1 + 4, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2)
    ax.imshow(im[:, :, ::-1]); ax.set_title(p.name, fontsize=8); ax.axis('off')
for ax in axes.flatten()[len(samples):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Training v2

Augmentasi dikumpulkan di `AUG` pada sel konfigurasi. Alasan tiap nilai:

| Parameter | Nilai | Alasan |
| :--- | :--- | :--- |
| `fliplr` | 0.0 | Uang tidak pernah terlihat seperti bayangan cermin, dan nominal yang tercermin membingungkan model. |
| `degrees` | 180 | **Perbaikan utama v2.2.** Bawaan ultralytics adalah `0.0`, sehingga v2.1 tidak pernah melihat uang miring. Pada uji 5 foto sendiri, tebakan berubah-ubah liar saat foto diputar: uang 1000 salah di 0 derajat tetapi benar di 90 derajat; uang 50000 salah di 0 derajat tetapi benar di tujuh sudut lain. Pengguna tunanetra tidak bisa membidik, jadi sudut acak adalah keadaan normal, bukan kasus tepi. |
| `perspective` | 0.0005 | Foto diambil dari sudut, bukan tegak lurus. Nilai kecil; terlalu besar membuat uang terlihat melengkung tidak wajar. |
| `scale` | 0.6 | Jarak kamera ke uang sangat bervariasi karena pengguna tidak melihat bingkainya. |
| `translate` | 0.2 | Uang sering tidak di tengah bingkai. |
| `hsv_v`, `hsv_s` | 0.5, 0.8 | Lampu kuning, siang terang, dan bayangan tangan menggeser warna. Uang 10000 emisi 2022 berwarna ungu-kebiruan dan mudah bergeser ke abu-abu di bawah cahaya hangat. |

`flipud` sengaja dibiarkan pada bawaannya (0.0): uang terbalik sudah tercakup oleh `degrees=180`.

Bandingkan hasilnya dengan v2.1 lewat tabel per kelas di Bagian 10. Perhatikan khusus kelas **2000** dan **10000**: pada uji foto sendiri, 2000 selalu dijawab 50000 di ke-13 sudut yang dicoba, dan 10000 tidak pernah benar. Bila setelah augmentasi rotasi kedua kelas itu masih buruk, penyebabnya bukan rotasi melainkan label; jalankan audit label di Bagian 10.3.


In [ ]:
model = YOLO(MODEL_WEIGHTS)
model.train(
    data=str(V2_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED,
    project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True, save_period=10, plots=True,
    **AUG,
)
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
print('Augmentasi yang dipakai:', AUG)

In [ ]:
# Jika sesi terputus di tengah training (hapus tanda #):
# RUN_DIR = RUNS_DIR / RUN_NAME
# model = YOLO(str(RUN_DIR / 'weights' / 'last.pt'))
# model.train(resume=True)
# BEST_PT = RUN_DIR / 'weights' / 'best.pt'

## 10. Evaluasi pada split test v2

In [ ]:
model = YOLO(str(BEST_PT))
test_metrics = model.val(data=str(V2_YAML), split='test', project=str(RUNS_DIR),
                         name=RUN_NAME + '_test', exist_ok=True)
print(f'v2 pada test v2   : mAP@0.5 = {test_metrics.box.map50:.3f} | mAP@0.5:0.95 = {test_metrics.box.map:.3f}')

idx = list(test_metrics.box.ap_class_index)
per_class = pd.DataFrame({
    'kelas': [CLASSES[i] for i in idx], 'precision': test_metrics.box.p, 'recall': test_metrics.box.r,
    'AP@0.5': test_metrics.box.ap50, 'AP@0.5:0.95': test_metrics.box.ap,
}).sort_values('AP@0.5')
display(per_class.round(3))

cm = Path(test_metrics.save_dir) / 'confusion_matrix.png'
if cm.exists():
    display(IPImage(filename=str(cm), width=800))

if BASELINE_PT and Path(BASELINE_PT).exists():
    base = YOLO(str(BASELINE_PT))
    base_metrics = base.val(data=str(V2_YAML), split='test', project=str(RUNS_DIR),
                            name=RUN_NAME + '_test_baseline', exist_ok=True, plots=False)
    print(f'\nBaseline pada test v2: mAP@0.5 = {base_metrics.box.map50:.3f}  (pembanding)')

### 10.1 Akurasi total nominal per foto

In [ ]:
test_imgs = sorted(p for p in (V2_ROOT / 'test' / 'images').iterdir())
rows = []
for img in test_imgs:
    gt = [CLASSES[int(l.split()[0])] for l in (V2_ROOT / 'test' / 'labels' / f'{img.stem}.txt').read_text().splitlines() if l.strip()]
    res = model.predict(source=str(img), conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)[0]
    pred_cls = res.boxes.cls.cpu().numpy().astype(int)
    pred = [CLASSES[c] for c in pred_cls]
    keep = ~nested_mask(list(pred_cls), res.boxes.xyxy.cpu().numpy(), NESTED_CONTAIN) if len(pred_cls) else []
    pred_f = [p for p, k in zip(pred, keep) if k]
    rows.append({'file': img.name, 'dataset': img.name[0],
                 'total_benar': sum(map(int, gt)), 'total_prediksi': sum(map(int, pred)),
                 'total_prediksi_saring': sum(map(int, pred_f)),
                 'lembar_benar': len(gt), 'lembar_prediksi': len(pred)})
eval_df = pd.DataFrame(rows)
eval_df['tepat_mentah'] = eval_df.total_benar == eval_df.total_prediksi
eval_df['tepat'] = eval_df.total_benar == eval_df.total_prediksi_saring
print(f'Akurasi total nominal (mentah)                 : {eval_df.tepat_mentah.mean():.1%}')
print(f'Akurasi total nominal (kotak bersarang disaring): {eval_df.tepat.mean():.1%} ({eval_df.tepat.sum()}/{len(eval_df)})')
print('Kolom tepat di bawah memakai versi disaring, sama seperti logika pipeline (PRD 8.6).')
display(eval_df.groupby('lembar_benar').tepat.agg(['mean', 'count']).round(3))
display(eval_df.groupby('dataset').tepat.agg(['mean', 'count']).round(3))
eval_df.to_csv(RUNS_DIR / f'{RUN_NAME}_test_totals.csv', index=False)

### 10.2 Galeri kesalahan

In [ ]:
wrong = eval_df[~eval_df.tepat].head(12)
if wrong.empty:
    print('Tidak ada kesalahan total nominal.')
else:
    fig, axes = plt.subplots((len(wrong) + 2) // 3, 3, figsize=(15, 4.8 * ((len(wrong) + 2) // 3)))
    axes = np.atleast_1d(axes).flatten()
    for ax, (_, r) in zip(axes, wrong.iterrows()):
        res = model.predict(source=str(V2_ROOT / 'test' / 'images' / r.file), conf=CONF_THRESHOLD,
                            imgsz=IMGSZ, verbose=False)[0]
        ax.imshow(res.plot()[:, :, ::-1])
        ax.set_title(f'{r.file}\nbenar {r.total_benar:,} | prediksi (disaring) {r.total_prediksi_saring:,}', fontsize=9)
        ax.axis('off')
    for ax in axes[len(wrong):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

### 10.3 Audit label v2

Mencari objek yang labelnya berbeda dari prediksi model yang sangat yakin, di semua split v2. Hasilnya daftar kandidat untuk dicek, bukan kepastian.

In [ ]:
AUDIT_CONF, AUDIT_IOU = 0.70, 0.5
audit = []
for split in ('train', 'val', 'test'):
    for img in sorted((V2_ROOT / split / 'images').iterdir()):
        lines = [l.split() for l in (V2_ROOT / split / 'labels' / f'{img.stem}.txt').read_text().splitlines() if l.strip()]
        if not lines:
            continue
        gt_cls = [int(l[0]) for l in lines]
        gt_box = xywh_to_xyxy([list(map(float, l[1:5])) for l in lines])
        res = model.predict(source=str(img), conf=AUDIT_CONF, imgsz=IMGSZ, verbose=False)[0]
        if len(res.boxes) == 0:
            continue
        pr_cls = res.boxes.cls.cpu().numpy().astype(int)
        ious = iou_matrix(gt_box, xywh_to_xyxy(res.boxes.xywhn.cpu().numpy()))
        for gi, gc in enumerate(gt_cls):
            pj = int(ious[gi].argmax())
            if ious[gi, pj] >= AUDIT_IOU and pr_cls[pj] != gc:
                audit.append({'split': split, 'file': img.name, 'label': CLASSES[gc],
                              'prediksi': CLASSES[pr_cls[pj]], 'conf': round(float(res.boxes.conf[pj]), 2)})
audit_df = pd.DataFrame(audit)
print(f'{len(audit_df)} kandidat label salah')
if len(audit_df):
    display(audit_df.groupby(['label', 'prediksi']).size().sort_values(ascending=False).rename('jumlah').to_frame())
    audit_df.merge(manifest_df, left_on='file', right_on='file_v2', how='left') \
            .to_csv(RUNS_DIR / f'{RUN_NAME}_audit_label.csv', index=False)

## 11. Simpan model dan catat keterbatasan

**Keterbatasan yang wajib ditulis di laporan:**
- Split test v2 berasal dari sumber yang sama dengan data latih, sehingga hasilnya cenderung optimistis. Pengujian final tetap membutuhkan foto sendiri.
- Sebagian besar data v2 berasal dari dataset B. Periksa apakah emisi 2016 cukup terwakili. Jika tidak, performa pada emisi 2016 perlu diuji terpisah.
- Dataset B tidak berisi foto tanpa uang, sehingga contoh latar hanya berasal dari sedikit gambar dataset A.

In [ ]:
stamp = datetime.now().strftime('%Y%m%d_%H%M')
dst = MODELS_DIR / f'best_money_{RUN_NAME}_{stamp}.pt'
shutil.copy(BEST_PT, dst)
print('Model disalin ke', dst)